# Notebook 02 — Preprocessing & Label Consolidation

**What this notebook does:**
1. Loads the 200k sample from Google Drive
2. **Consolidates product labels** — CFPB renamed categories over time, leaving duplicate class names that must be merged before modeling
3. **Cleans complaint text** — removes redaction markers, HTML, boilerplate, excess whitespace
4. **Splits into train / val / test** (70 / 15 / 15) stratified by product
5. Saves three clean CSVs to Drive for use by all downstream notebooks

**Why label consolidation matters:**
Without this step the model sees 21 classes — but many are the same product under a different name (CFPB changed labels in 2017 and 2020). A model trained on raw labels would learn to distinguish naming conventions, not actual products.

## Cell 1 — Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = '/content/drive/MyDrive/nlp_project'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

NARRATIVE_COL = 'consumer_complaint_narrative'
LABEL_COL     = 'product_clean'   # our consolidated label column

print('Ready. Data dir:', DATA_DIR)

## Cell 2 — Load Sample from Drive

In [ ]:
IN_PATH = os.path.join(DATA_DIR, 'complaints_sample.csv')
df = pd.read_csv(IN_PATH, low_memory=False)

print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
print(f'\nRaw product value counts:')
print(df['product'].value_counts().to_string())

## Cell 3 — Product Label Consolidation

CFPB has renamed its product taxonomy twice (2017, 2020), leaving duplicate labels in the historical data.
We map all 21 raw labels to 8 clean canonical classes.

**Mapping rationale:**
- All 3 credit reporting variants → `Credit Reporting` (they cover the same complaints)
- Credit card + prepaid card variants → `Credit Card`
- All payday/title/personal loan variants + student + vehicle + consumer → `Loans`
- Checking/savings + bank account → `Banking`
- All money transfer / virtual currency variants → `Money Transfer`
- `Debt or credit management` and `Other financial service` → `Other` (too few records to model alone)

In [ ]:
PRODUCT_MAP = {
    # Credit Reporting — 3 CFPB-era names for the same category
    'Credit reporting':                                                          'Credit Reporting',
    'Credit reporting or other personal consumer reports':                       'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',

    # Debt Collection
    'Debt collection':                                                           'Debt Collection',

    # Credit Card
    'Credit card':                                                               'Credit Card',
    'Credit card or prepaid card':                                               'Credit Card',
    'Prepaid card':                                                              'Credit Card',

    # Banking
    'Bank account or service':                                                   'Banking',
    'Checking or savings account':                                               'Banking',

    # Mortgage
    'Mortgage':                                                                  'Mortgage',

    # Loans — student, vehicle, payday, personal, consumer all under one umbrella
    'Student loan':                                                              'Loans',
    'Vehicle loan or lease':                                                     'Loans',
    'Consumer Loan':                                                             'Loans',
    'Payday loan':                                                               'Loans',
    'Payday loan, title loan, or personal loan':                                 'Loans',
    'Payday loan, title loan, personal loan, or advance loan':                   'Loans',

    # Money Transfer
    'Money transfers':                                                           'Money Transfer',
    'Money transfer, virtual currency, or money service':                        'Money Transfer',
    'Virtual currency':                                                          'Money Transfer',

    # Other — too few records to model as separate classes
    'Debt or credit management':                                                 'Other',
    'Other financial service':                                                   'Other',
}

df[LABEL_COL] = df['product'].map(PRODUCT_MAP)

# Check for any unmapped labels
unmapped = df[df[LABEL_COL].isna()]['product'].unique()
if len(unmapped):
    print(f'WARNING — unmapped product labels: {unmapped}')
else:
    print('All product labels mapped successfully.')

print(f'\nConsolidated class distribution:')
counts = df[LABEL_COL].value_counts()
print(counts.to_string())
print(f'\nTotal classes : {counts.shape[0]}')
print(f'Imbalance ratio (max/min): {counts.max()/counts.min():.1f}x')

In [ ]:
# Drop the tiny 'Other' class — 282 records across 200k won't generalise
before = len(df)
df = df[df[LABEL_COL] != 'Other'].copy()
print(f'Dropped {before - len(df)} rows labelled Other. Remaining: {len(df):,}')
print(f'Final class counts:')
print(df[LABEL_COL].value_counts().to_string())

# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
df[LABEL_COL].value_counts().sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white'
)
ax.set_title('Consolidated Product Classes', fontsize=13)
ax.set_xlabel('Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'consolidated_classes.png'), dpi=120)
plt.show()

## Cell 4 — Text Cleaning

The CFPB narrative text has specific noise patterns we need to handle:

| Pattern | Example | Action |
|---|---|---|
| Redaction markers | `XXXX`, `XX/XX/XXXX` | Remove — uninformative filler |
| HTML entities | `&amp;`, `&lt;` | Unescape |
| Boilerplate sign-off | `Sincerely, ...` | Remove |
| Extra whitespace | `  ` | Normalise to single space |
| Newlines / tabs | `\n`, `\t` | Replace with space |

We do **not** lowercase — we preserve case for BERT (which uses cased models and benefits from capitalisation signals like proper nouns).

In [ ]:
import html

def clean_narrative(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # Unescape HTML entities (&amp; → &)
    text = html.unescape(text)
    # Remove CFPB redaction markers: XXXX, XX/XX/XXXX, XX/XXXX, etc.
    text = re.sub(r'X{2,}(?:/X{2,})*', '', text)
    # Remove newlines and tabs
    text = re.sub(r'[\n\r\t]', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

# Apply
df['narrative_clean'] = df[NARRATIVE_COL].apply(clean_narrative)

# Sanity check — show before/after for 3 examples
print('--- Before / After cleaning examples ---\n')
for idx in df.sample(3, random_state=1).index:
    raw   = df.loc[idx, NARRATIVE_COL]
    clean = df.loc[idx, 'narrative_clean']
    print(f'RAW   : {raw[:200]}')
    print(f'CLEAN : {clean[:200]}')
    print()

In [ ]:
# Drop rows where cleaning left an empty or very short narrative (< 10 chars)
before = len(df)
df = df[df['narrative_clean'].str.len() >= 10].copy()
print(f'Dropped {before - len(df)} near-empty rows after cleaning. Remaining: {len(df):,}')

# Update narrative_len on the cleaned text
df['narrative_len'] = df['narrative_clean'].str.count(' ') + 1
print(f'\nCleaned narrative word count stats:')
print(df['narrative_len'].describe(percentiles=[.5, .75, .95, .99]).to_string())

## Cell 5 — Label Encoding

Models need integer labels. We create a stable mapping and save it so every notebook uses the same encoding.

In [ ]:
# Sorted so the mapping is deterministic across runs
classes    = sorted(df[LABEL_COL].unique())
label2id   = {c: i for i, c in enumerate(classes)}
id2label   = {i: c for c, i in label2id.items()}

df['label'] = df[LABEL_COL].map(label2id)

print('Label encoding:')
for name, idx in label2id.items():
    print(f'  {idx}  {name}')

# Save the mapping to Drive so all notebooks share the same encoding
import json
mapping_path = os.path.join(DATA_DIR, 'label_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)
print(f'\nSaved label mapping to: {mapping_path}')

## Cell 6 — Train / Val / Test Split

We split **stratified by label** so every class appears in all three sets at the same proportion.

- **Train (70%)** — model fitting
- **Val (15%)** — hyperparameter tuning, early stopping
- **Test (15%)** — held-out final evaluation (touch only once)

The test set is locked away — we won't look at it until Notebook 06 (Evaluation).

In [ ]:
from sklearn.model_selection import train_test_split

# First split off test (15%)
df_trainval, df_test = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42
)

# Then split train/val from the remaining 85%  (val = 15/85 ≈ 17.6% of trainval)
df_train, df_val = train_test_split(
    df_trainval, test_size=(0.15 / 0.85), stratify=df_trainval['label'], random_state=42
)

print(f'Train : {len(df_train):>7,} rows  ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(df_val):>7,} rows  ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(df_test):>7,} rows  ({len(df_test)/len(df)*100:.1f}%)')

print('\nClass distribution in train:')
print(df_train[LABEL_COL].value_counts().to_string())

## Cell 7 — Compute Class Weights

Credit Reporting dominates at ~66% of data. If we train without correction, the model learns to
predict Credit Reporting for almost everything and gets high accuracy but terrible F1 on minority classes.

We compute sklearn's `balanced` class weights now and save them. XGBoost and the Transformer will both use these.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.array(sorted(label2id.values())),
    y=df_train['label'].values
)
class_weights = dict(enumerate(cw))

print('Class weights (higher = model penalised more for missing this class):')
for idx, w in class_weights.items():
    print(f'  {idx}  {id2label[idx]:<45}  weight: {w:.3f}')

# Save
cw_path = os.path.join(DATA_DIR, 'class_weights.json')
with open(cw_path, 'w') as f:
    json.dump({str(k): v for k, v in class_weights.items()}, f, indent=2)
print(f'\nSaved class weights to: {cw_path}')

## Cell 8 — Save Splits to Drive

We save lean versions of each split — only the columns downstream notebooks need:
- `complaint_id` — for traceability
- `narrative_clean` — the cleaned input text
- `label` — integer label
- `product_clean` — human-readable label (useful for plots)
- `narrative_len`, `company`, `state`, `tags`, `submitted_via` — metadata features for Notebook 03

In [ ]:
SAVE_COLS = [
    'complaint_id', 'narrative_clean', 'label', LABEL_COL,
    'narrative_len', 'company', 'state', 'tags', 'submitted_via'
]
save_cols = [c for c in SAVE_COLS if c in df_train.columns]

splits = {'train': df_train, 'val': df_val, 'test': df_test}

for name, split_df in splits.items():
    path = os.path.join(DATA_DIR, f'{name}.csv')
    split_df[save_cols].to_csv(path, index=False)
    size_mb = os.path.getsize(path) / 1e6
    print(f'{name:5s} → {path}  ({len(split_df):,} rows, {size_mb:.1f} MB)')

## Cell 9 — Summary (paste this output to Claude)

In [ ]:
print('='*60)
print('  NOTEBOOK 02 SUMMARY — paste this output to Claude')
print('='*60)
print(f'Total rows after cleaning         : {len(df):,}')
print(f'Classes after consolidation       : {len(classes)}')
print(f'Class names                       : {classes}')
print()
print('Final class distribution (full dataset):')
print(df[LABEL_COL].value_counts().to_string())
print()
print(f'Train size                        : {len(df_train):,}')
print(f'Val size                          : {len(df_val):,}')
print(f'Test size                         : {len(df_test):,}')
print()
print('Class weights:')
for idx, w in class_weights.items():
    print(f'  {id2label[idx]:<45}  {w:.3f}')
print()
print(f'Narrative len median (clean)      : {df["narrative_len"].median():.0f} words')
print(f'Narrative len 95th pct (clean)    : {df["narrative_len"].quantile(0.95):.0f} words')
print(f'Files saved to Drive              : train.csv, val.csv, test.csv, label_mapping.json, class_weights.json')
print('='*60)